# Step 1 — Code Pattern Analysis for Significantly Confused CWE Pairs

**Goal:** For each pair the model frequently confuses, extract regex-based code features from
the raw source code and compare their frequencies between the two classes.
Features with high |Δ| are signals the model *should* be using but isn't.
Features near Δ=0 are ambiguous — likely the root cause of confusion.

**Inputs:**
- `juliet_cwe_dataset.csv` — full dataset (1.4 GB, read in chunks)
- `cwe_confusion_analysis.csv` — significant confused pairs

**Columns used:** `cwe`, `code_clean`

## 1. Imports & Config

In [5]:
import pandas as pd
import numpy as np
import re
import gc

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_FILE   = "juliet_cwe_dataset.csv"      # full 1.4 GB file
# DATASET_FILE = "juliet_cwe_sample_1__.csv"   # ← swap to this for quick testing

CONFUSION_FILE = "cwe_confusion_analysis.csv"
CODE_COL       = "code_clean"
CWE_COL        = "cwe"

CHUNK_SIZE     = 5_000   # lower to 2_000 if memory is still tight
# ──────────────────────────────────────────────────────────────────────────────
print("✅ Config ready")

✅ Config ready


## 2. Load Confused Pairs

We only analyse pairs that appear in `cwe_confusion_analysis.csv`.  
Both `True_CWE` and `Predicted_CWE` are kept — each direction is a separate failure mode.

In [6]:
conf_df = pd.read_csv(CONFUSION_FILE)

# Safety: drop rows where both sides are the same (diagonal)
conf_df = conf_df[conf_df["True_CWE"] != conf_df["Predicted_CWE"]].reset_index(drop=True)

confused_cwes = set(conf_df["True_CWE"]) | set(conf_df["Predicted_CWE"])

print(f"Confused pairs : {len(conf_df)}")
print(f"Unique CWEs    : {len(confused_cwes)} → {sorted(confused_cwes)}")
print()
print(conf_df[["True_CWE", "Predicted_CWE", "Count", "Confusion_Rate"]].to_string(index=False))

Confused pairs : 16
Unique CWEs    : 13 → ['CWE114', 'CWE121', 'CWE122', 'CWE123', 'CWE124', 'CWE126', 'CWE127', 'CWE134', 'CWE15', 'CWE176', 'CWE188', 'CWE190', 'CWE244']

True_CWE Predicted_CWE  Count  Confusion_Rate
  CWE188        CWE190     10        0.013514
  CWE176        CWE190      8        0.010811
  CWE121        CWE123      6        0.002158
  CWE122        CWE123      6        0.001660
  CWE114        CWE127      4        0.009009
  CWE121        CWE190      2        0.000719
  CWE122        CWE190      2        0.000553
  CWE123         CWE15      2        0.001252
  CWE124        CWE190      2        0.002105
  CWE126         CWE15      2        0.001252
  CWE134         CWE15      2        0.000644
  CWE134        CWE190      2        0.000644
   CWE15        CWE190      2        0.001152
  CWE176         CWE15      2        0.002703
  CWE188         CWE15      2        0.002703
  CWE244         CWE15      2        0.004386


## 3. CWE-Targeted Feature Library

Features are designed from actual Juliet source code for the specific CWEs present in the
confused pairs. Each pattern is a compiled regex; matching is binary (present / not present).

Groups map directly to the confused-pair families:

| Group | Relevant confused pairs |
|---|---|
| `integer_overflow` | CWE188 ↔ CWE190, CWE176 ↔ CWE190, CWE124 ↔ CWE190 |
| `buffer_ops` | CWE121 ↔ CWE123, CWE121 ↔ CWE190, CWE127 ↔ CWE114 |
| `pointer_struct` | CWE188 ↔ CWE190, CWE123 ↔ CWE15 |
| `unicode_wchar` | CWE176 ↔ CWE190, CWE176 ↔ CWE15 |
| `process_control` | CWE114 ↔ CWE127 |
| `memory_mgmt` | general across all pairs |
| `control_flow` | general across all pairs |
| `error_handling` | general across all pairs |

In [7]:
_RAW = [

    # ── INTEGER OVERFLOW (CWE190, CWE188, CWE176, CWE124) ─────────────────────
    # CWE190 hallmarks: integer arithmetic on large types, overflow guards
    ("int64_type",           r'\bint64_t\b',                              "integer_overflow"),
    ("LLONG_MAX_used",       r'\bLLONG_MAX\b',                           "integer_overflow"),
    ("INT_MAX_used",         r'\bINT_MAX\b',                              "integer_overflow"),
    ("overflow_guard",       r'if\s*\([^)]*<\s*(?:LLONG_MAX|INT_MAX|UINT_MAX)[^)]*\)', "integer_overflow"),
    ("multiply_op",          r'\w+\s*\*\s*\w+',                           "integer_overflow"),
    ("add_op_result",        r'\w+\s*\+\s*\w+\s*;',                       "integer_overflow"),
    ("cast_to_int",          r'\(\s*int\s*\)',                            "integer_overflow"),
    ("cast_to_long",         r'\(\s*(long|int64_t|int32_t)\s*\)',         "integer_overflow"),
    ("unsigned_type",        r'\bunsigned\s+(int|long|short|char)\b',     "integer_overflow"),
    ("signed_comparison",    r'(data|result)\s*[<>]\s*0',                 "integer_overflow"),
    ("hex_literal",          r'0[xX][0-9a-fA-F]+',                        "integer_overflow"),
    ("LL_literal",           r'\d+LL\b',                                  "integer_overflow"),

    # ── STRUCT / POINTER MEMORY LAYOUT (CWE188 specific) ──────────────────────
    # CWE188: reliance on struct memory layout — cast pointer across struct fields
    ("struct_def",           r'\bstruct\s+\w*\s*\{',                      "pointer_struct"),
    ("struct_ptr_cast",      r'\(\s*\w+\s*\*\s*\)\s*\(',                  "pointer_struct"),
    ("ptr_cast_arith",       r'\(\s*int\s*\*\s*\)\s*\(\s*\w+\s*\+',       "pointer_struct"),
    ("deref_cast_ptr",       r'\*\s*\(\s*\w+\s*\*\s*\)',                  "pointer_struct"),
    ("struct_member_access", r'\w+\.\w+\s*=',                             "pointer_struct"),
    ("sizeof_in_ptr_arith",  r'\+\s*sizeof\s*\(',                         "pointer_struct"),
    ("address_of_member",    r'&\s*\w+\.\w+',                             "pointer_struct"),

    # ── UNICODE / WCHAR (CWE176 specific) ─────────────────────────────────────
    # CWE176: improper handling of unicode — wchar_t, wcscpy, unicode escapes
    ("wchar_type",           r'\bwchar_t\b',                              "unicode_wchar"),
    ("wcscpy_present",       r'\bwcscpy\s*\(',                            "unicode_wchar"),
    ("wcsncpy_present",      r'\bwcsncpy\s*\(',                           "unicode_wchar"),
    ("wmemset_present",      r'\bwmemset\s*\(',                           "unicode_wchar"),
    ("wcslen_present",       r'\bwcslen\s*\(',                            "unicode_wchar"),
    ("unicode_escape",       r'\\u[0-9a-fA-F]{4}',                        "unicode_wchar"),
    ("L_string_literal",     r'L"[^"]*"',                                 "unicode_wchar"),
    ("L_char_literal",       r"L'[^']+'",                                 "unicode_wchar"),
    ("printWLine",           r'\bprintWLine\s*\(',                         "unicode_wchar"),

    # ── BUFFER / STACK OVERFLOW (CWE121, CWE122, CWE124, CWE127) ─────────────
    # CWE121: stack-based overflow (ALLOCA, fixed-size array, loop write)
    # CWE122: heap-based overflow (malloc/calloc + loop write)
    # CWE124: buffer underwrite
    # CWE127: buffer underread
    ("alloca_used",          r'\bALLOCA\s*\(',                            "buffer_ops"),
    ("malloc_used",          r'\bmalloc\s*\(',                            "buffer_ops"),
    ("calloc_used",          r'\bcalloc\s*\(',                            "buffer_ops"),
    ("realloc_used",         r'\brealloc\s*\(',                           "buffer_ops"),
    ("free_used",            r'\bfree\s*\(',                              "buffer_ops"),
    ("stack_char_array",     r'\bchar\s+\w+\s*\[\s*\d+\s*\]',             "buffer_ops"),
    ("stack_wchar_array",    r'\bwchar_t\s+\w+\s*\[\s*\d+\s*\]',          "buffer_ops"),
    ("fixed_int_array",      r'\bint\w*\s*\w+\s*\[\s*\d+\s*\]',           "buffer_ops"),
    ("ptr_minus_offset",     r'\w+\s*-\s*\d+\s*;',                        "buffer_ops"),  # underread/underwrite
    ("ptr_plus_offset",      r'\w+\s*\+\s*\d+\s*;',                       "buffer_ops"),
    ("loop_index_write",     r'for\s*\([^)]*\)\s*\{[^}]*\w+\s*\[\s*\w',   "buffer_ops"),
    ("memcpy_used",          r'\bmemcpy\s*\(',                            "buffer_ops"),
    ("memmove_used",         r'\bmemmove\s*\(',                           "buffer_ops"),
    ("memset_used",          r'\bmemset\s*\(',                            "buffer_ops"),
    ("sizeof_used",          r'\bsizeof\s*\(',                            "buffer_ops"),
    ("strncpy_used",         r'\bstrncpy\s*\(',                           "buffer_ops"),
    ("strcpy_used",          r'\bstrcpy\s*\(',                            "buffer_ops"),
    ("dataBadBuffer",        r'dataBadBuffer',                             "buffer_ops"),  # Juliet naming convention
    ("dataGoodBuffer",       r'dataGoodBuffer',                            "buffer_ops"),

    # ── WRITE-WHAT-WHERE (CWE123 specific) ────────────────────────────────────
    # CWE123: linked list / arbitrary write via corrupted pointer
    ("linkedList_struct",    r'linkedList|linked_list',                    "write_what_where"),
    ("prev_next_ptr",        r'\b(prev|next)\s*;',                        "write_what_where"),
    ("struct_ptr_assign",    r'\w+\s*->\s*(next|prev)\s*=',               "write_what_where"),
    ("arbitrary_write",      r'\*\s*\w+\s*=\s*\w+',                       "write_what_where"),

    # ── PROCESS CONTROL (CWE114 specific) ─────────────────────────────────────
    # CWE114: LoadLibrary / dlopen with user-controlled path
    ("LoadLibrary_call",     r'\bLoadLibrary\w*\s*\(',                    "process_control"),
    ("dlopen_call",          r'\bdlopen\s*\(',                            "process_control"),
    ("GetProcAddress",       r'\bGetProcAddress\s*\(',                    "process_control"),
    ("socket_source",        r'recv\s*\(',                                "process_control"),  # tainted input via socket
    ("env_source",           r'\bgetenv\s*\(',                            "process_control"),
    ("replace_str",          r'\breplace\b',                              "process_control"),  # CWE114 Juliet uses 'replace'

    # ── IMPROPER CHECK / NUMERIC TRUNCATION (CWE15, CWE134) ───────────────────
    ("printf_used",          r'\bprintf\s*\(',                            "format_check"),
    ("sprintf_used",         r'\bsprintf\s*\(',                           "format_check"),
    ("snprintf_used",        r'\bsnprintf\s*\(',                          "format_check"),
    ("format_spec_n",        r'%n',                                        "format_check"),
    ("format_spec_s",        r'%s',                                        "format_check"),
    ("extern_data_source",   r'\b(fscanf|scanf|fgets|recv|read)\s*\(',    "format_check"),
    ("environment_var",      r'\bgetenv\s*\(',                            "format_check"),

    # ── MEMORY MANAGEMENT (general) ───────────────────────────────────────────
    ("null_check",           r'(==|!=)\s*(NULL|nullptr)',                  "memory_mgmt"),
    ("null_assign",          r'=\s*(NULL|nullptr)\s*;',                    "memory_mgmt"),
    ("double_free_risk",     r'free\s*\([^)]+\)[^;]*;[^}]*free\s*\(',     "memory_mgmt"),
    ("new_operator",         r'\bnew\b',                                   "memory_mgmt"),
    ("delete_operator",      r'\bdelete\b',                               "memory_mgmt"),
    ("uses_namespace",       r'\bnamespace\b',                            "memory_mgmt"),  # C++ indicator

    # ── CONTROL FLOW (general) ─────────────────────────────────────────────────
    ("has_for_loop",         r'\bfor\s*\(',                               "control_flow"),
    ("has_while_loop",       r'\bwhile\s*\(',                             "control_flow"),
    ("GLOBAL_CONST_TRUE",    r'GLOBAL_CONST_TRUE',                         "control_flow"),  # Juliet taint flag
    ("globalReturnsTrue",    r'globalReturnsTrue\s*\(',                    "control_flow"),
    ("OMITBAD_guard",        r'OMITBAD',                                   "control_flow"),
    ("OMITGOOD_guard",       r'OMITGOOD',                                  "control_flow"),
    ("static_five_cond",     r'if\s*\(\s*5\s*==\s*5\s*\)',               "control_flow"),  # Juliet dead-code pattern
    ("early_return",         r'\breturn\b',                               "control_flow"),
]

# Compile all patterns once
FEATURES       = [(n, re.compile(p, re.DOTALL), g) for n, p, g in _RAW]
FEATURE_NAMES  = [f[0] for f in FEATURES]
FEATURE_GROUPS = {f[0]: f[2] for f in FEATURES}

print(f"✅ {len(FEATURES)} patterns compiled across "
      f"{len(set(f[2] for f in FEATURES))} groups")
for grp in dict.fromkeys(f[2] for f in FEATURES):
    n = sum(1 for f in FEATURES if f[2] == grp)
    print(f"   {grp:<20} {n} patterns")

✅ 78 patterns compiled across 9 groups
   integer_overflow     12 patterns
   pointer_struct       7 patterns
   unicode_wchar        9 patterns
   buffer_ops           19 patterns
   write_what_where     4 patterns
   process_control      6 patterns
   format_check         7 patterns
   memory_mgmt          6 patterns
   control_flow         8 patterns


## 4. Chunked Feature Extraction

Reads the full CSV in chunks of `CHUNK_SIZE` rows.  
Each chunk is filtered to only the CWEs involved in confused pairs before any regex work —  
so memory stays flat regardless of total file size.

Accumulates `(hit_count, total_count)` per `(CWE, feature)` then discards the chunk.

In [8]:
# Accumulators: cwe → feature → [hits, total]
accum = {cwe: {feat: [0, 0] for feat in FEATURE_NAMES} for cwe in confused_cwes}

chunk_num  = 0
total_kept = 0

for chunk in pd.read_csv(
    DATASET_FILE,
    usecols=[CWE_COL, CODE_COL],          # load only the two columns we need
    chunksize=CHUNK_SIZE,
    dtype={CWE_COL: str, CODE_COL: str},
    na_filter=False,                       # blanks → empty string, not NaN
):
    chunk_num += 1

    # Filter immediately — drop rows for CWEs we don't care about
    chunk = chunk[chunk[CWE_COL].isin(confused_cwes)]
    if chunk.empty:
        continue

    total_kept  += len(chunk)
    code_series  = chunk[CODE_COL]
    cwe_series   = chunk[CWE_COL]

    for feat_name, compiled_pat, _ in FEATURES:
        # Vectorised match across the whole chunk column — no Python loop per row
        hits = code_series.str.contains(compiled_pat, regex=True, na=False)
        for cwe in cwe_series.unique():
            mask = cwe_series == cwe
            accum[cwe][feat_name][0] += int(hits[mask].sum())
            accum[cwe][feat_name][1] += int(mask.sum())

    del chunk, code_series, cwe_series, hits
    gc.collect()

    if chunk_num % 10 == 0:
        print(f"  chunk {chunk_num:>4}  |  kept so far: {total_kept:,}")

print(f"\n✅ Done — {chunk_num} chunks, {total_kept:,} rows kept")

/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expre

  chunk   20  |  kept so far: 37,511


/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)
/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expre

  chunk   30  |  kept so far: 67,381


/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)



✅ Done — 36 chunks, 71,524 rows kept


/var/folders/_t/79359yk13vl8dqw3v2p08c9c0000gn/T/ipykernel_45485/3735622923.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = code_series.str.contains(compiled_pat, regex=True, na=False)


## 5. Per-CWE Feature Frequency Table

`freq[CWE][feature]` = fraction of samples in that CWE class where the pattern is present.  
e.g. `0.92` means 92% of CWE190 samples contain that pattern.

In [9]:
freq_rows = []
for cwe, feat_dict in accum.items():
    row = {CWE_COL: cwe}
    for feat, (hits, total) in feat_dict.items():
        row[feat] = round(hits / total, 4) if total > 0 else 0.0
    freq_rows.append(row)

cwe_freq = (
    pd.DataFrame(freq_rows)
    .set_index(CWE_COL)
    .sort_index()
)

# Sample counts per CWE
sample_counts = {cwe: accum[cwe][FEATURE_NAMES[0]][1] for cwe in confused_cwes}

print(f"Frequency table shape: {cwe_freq.shape}  (CWEs × features)")
print()
print("Samples per CWE:")
for cwe, n in sorted(sample_counts.items()):
    print(f"  {cwe:<10} {n:>6} samples")

Frequency table shape: (13, 78)  (CWEs × features)

Samples per CWE:
  CWE114       1780 samples
  CWE121      14938 samples
  CWE122      16314 samples
  CWE123        462 samples
  CWE124       5954 samples
  CWE126       4156 samples
  CWE127       5954 samples
  CWE134       9150 samples
  CWE15         152 samples
  CWE176        156 samples
  CWE188         78 samples
  CWE190      12282 samples
  CWE244        148 samples


## 6. Pairwise Pattern Comparison

For each confused pair `(True_CWE A → Predicted_CWE B)`:

$$\Delta = \text{freq}(A) - \text{freq}(B)$$

| |Δ| range | Interpretation |
|---|---|
| ≥ 0.20 | **Strongly discriminating** — clear signal the model should catch |
| 0.10–0.20 | **Moderately discriminating** — useful but subtle |
| < 0.05 | **Ambiguous** — classes look identical here, likely confusion trigger |

In [10]:
master_rows    = []
pair_summaries = {}
SEP  = "═" * 76
SEP2 = "─" * 76

for _, pr in conf_df.iterrows():
    cwe_a = pr["True_CWE"]
    cwe_b = pr["Predicted_CWE"]

    if cwe_a not in cwe_freq.index or cwe_b not in cwe_freq.index:
        print(f"  ⚠️  Skipping ({cwe_a} → {cwe_b}): not found in dataset")
        continue

    fa    = cwe_freq.loc[cwe_a]
    fb    = cwe_freq.loc[cwe_b]
    delta = (fa - fb).round(4)

    pair_df = pd.DataFrame({
        "Feature":         FEATURE_NAMES,
        "Group":           [FEATURE_GROUPS[f] for f in FEATURE_NAMES],
        f"Freq_{cwe_a}":   fa.values.round(4),
        f"Freq_{cwe_b}":   fb.values.round(4),
        "Delta":           delta.values,
        "Abs_Delta":       delta.abs().values,
    }).sort_values("Abs_Delta", ascending=False).reset_index(drop=True)

    pair_summaries[(cwe_a, cwe_b)] = {
        "count":           int(pr["Count"]),
        "rate":            float(pr["Confusion_Rate"]),
        "n_a":             sample_counts.get(cwe_a, 0),
        "n_b":             sample_counts.get(cwe_b, 0),
        "strong_disc":     pair_df[pair_df["Abs_Delta"] >= 0.20],
        "moderate_disc":   pair_df[(pair_df["Abs_Delta"] >= 0.10) & (pair_df["Abs_Delta"] < 0.20)],
        "ambiguous":       pair_df[pair_df["Abs_Delta"] < 0.05],
        "pair_df":         pair_df,
    }

    for _, r in pair_df.iterrows():
        master_rows.append({
            "True_CWE":       cwe_a,
            "Predicted_CWE":  cwe_b,
            "Confusion_Count":int(pr["Count"]),
            "Confusion_Rate": float(pr["Confusion_Rate"]),
            "Feature":        r["Feature"],
            "Group":          r["Group"],
            f"Freq_{cwe_a}":  r[f"Freq_{cwe_a}"],
            f"Freq_{cwe_b}":  r[f"Freq_{cwe_b}"],
            "Delta":          r["Delta"],
            "Abs_Delta":      r["Abs_Delta"],
        })

master_df = pd.DataFrame(master_rows)
print(f"✅ {len(pair_summaries)} pairs analysed, master table: {len(master_df):,} rows")

✅ 16 pairs analysed, master table: 1,248 rows


## 7. Per-Pair Reports

In [11]:
for (cwe_a, cwe_b), info in pair_summaries.items():

    print(SEP)
    print(f"  {cwe_a}  →misclassified as→  {cwe_b}")
    print(f"  Confused {info['count']} times  |  error rate {info['rate']*100:.2f}%  "
          f"|  samples: {cwe_a}={info['n_a']}, {cwe_b}={info['n_b']}")
    print(SEP)

    # Strongly discriminating
    sd = info["strong_disc"]
    if not sd.empty:
        print(f"  ▶▶ STRONGLY discriminating (|Δ| ≥ 0.20)")
        print(f"     The model SHOULD be separating on these — investigate why it isn't:\n")
        for _, r in sd.iterrows():
            dom = cwe_a if r["Delta"] > 0 else cwe_b
            print(f"     [{r['Group']:<17}]  {r['Feature']:<25}  "
                  f"{cwe_a}={r[f'Freq_{cwe_a}']:.2f}  "
                  f"{cwe_b}={r[f'Freq_{cwe_b}']:.2f}  "
                  f"Δ={r['Delta']:+.3f}  ← {dom} dominates")

    # Moderately discriminating
    md = info["moderate_disc"]
    if not md.empty:
        print()
        print(f"  ▶  Moderately discriminating (0.10 ≤ |Δ| < 0.20):\n")
        for _, r in md.iterrows():
            dom = cwe_a if r["Delta"] > 0 else cwe_b
            print(f"     [{r['Group']:<17}]  {r['Feature']:<25}  "
                  f"{cwe_a}={r[f'Freq_{cwe_a}']:.2f}  "
                  f"{cwe_b}={r[f'Freq_{cwe_b}']:.2f}  "
                  f"Δ={r['Delta']:+.3f}")

    # Ambiguous
    amb = info["ambiguous"]
    if not amb.empty:
        print()
        print(f"  ✖  Ambiguous features (|Δ| < 0.05) — classes look identical here.")
        print(f"     These are the likely root cause of confusion:\n")
        for _, r in amb.head(8).iterrows():
            print(f"     [{r['Group']:<17}]  {r['Feature']:<25}  "
                  f"{cwe_a}={r[f'Freq_{cwe_a}']:.2f}  "
                  f"{cwe_b}={r[f'Freq_{cwe_b}']:.2f}")
    print()

════════════════════════════════════════════════════════════════════════════
  CWE188  →misclassified as→  CWE190
  Confused 10 times  |  error rate 1.35%  |  samples: CWE188=78, CWE190=12282
════════════════════════════════════════════════════════════════════════════
  ▶▶ STRONGLY discriminating (|Δ| ≥ 0.20)
     The model SHOULD be separating on these — investigate why it isn't:

     [pointer_struct   ]  struct_def                 CWE188=0.92  CWE190=0.03  Δ=+0.894  ← CWE188 dominates
     [pointer_struct   ]  struct_member_access       CWE188=0.92  CWE190=0.10  Δ=+0.826  ← CWE188 dominates
     [pointer_struct   ]  address_of_member          CWE188=0.46  CWE190=0.00  Δ=+0.462  ← CWE188 dominates
     [integer_overflow ]  hex_literal                CWE188=0.46  CWE190=0.00  Δ=+0.462  ← CWE188 dominates
     [pointer_struct   ]  sizeof_in_ptr_arith        CWE188=0.46  CWE190=0.00  Δ=+0.462  ← CWE188 dominates
     [pointer_struct   ]  struct_ptr_cast            CWE188=0.46  CWE190=0.

## 8. Global Feature Discriminability Ranking

Which features are most useful for separating confused pairs across the board?  
Low mean |Δ| = this feature looks the same in both classes everywhere = weak discriminator.

In [12]:
global_rank = (
    master_df
    .groupby(["Feature", "Group"])["Abs_Delta"]
    .mean()
    .reset_index()
    .rename(columns={"Abs_Delta": "Mean_Abs_Delta"})
    .sort_values("Mean_Abs_Delta", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 most discriminating features across all confused pairs:")
print(global_rank.head(15).to_string(index=False))
print()
print("Top 10 most ambiguous features (weakest discrimination):")
print(global_rank.tail(10).sort_values("Mean_Abs_Delta").to_string(index=False))

Top 15 most discriminating features across all confused pairs:
             Feature            Group  Mean_Abs_Delta
          wchar_type    unicode_wchar        0.360900
struct_member_access   pointer_struct        0.342394
    stack_char_array       buffer_ops        0.332163
  extern_data_source     format_check        0.321731
       socket_source  process_control        0.304688
      has_while_loop     control_flow        0.289875
         memset_used       buffer_ops        0.289006
         strcpy_used       buffer_ops        0.278156
          struct_def   pointer_struct        0.269781
         sizeof_used       buffer_ops        0.254656
    L_string_literal    unicode_wchar        0.228519
          null_check      memory_mgmt        0.225150
   stack_wchar_array       buffer_ops        0.211425
   linkedList_struct write_what_where        0.185062
       prev_next_ptr write_what_where        0.185062

Top 10 most ambiguous features (weakest discrimination):
         Featur

## 9. Export

In [13]:
# 1. Full (pair × feature) master table
master_df.to_csv("cwe_pattern_analysis_master.csv", index=False)

# 2. Per-CWE feature frequency matrix
cwe_freq.to_csv("cwe_feature_frequencies.csv")

# 3. Global discriminability ranking
global_rank.to_csv("cwe_feature_discriminability.csv", index=False)

# 4. Top-20 features per pair sorted by |Δ|
top_per_pair = pd.concat([
    info["pair_df"]
    .head(20)
    .assign(
        True_CWE=cwe_a,
        Predicted_CWE=cwe_b,
        Confusion_Count=info["count"],
        Confusion_Rate=info["rate"],
    )
    for (cwe_a, cwe_b), info in pair_summaries.items()
])
top_per_pair.to_csv("cwe_top_features_per_pair.csv", index=False)

print("✅ Exported:")
print("   cwe_pattern_analysis_master.csv   — full (pair × feature) comparison table")
print("   cwe_feature_frequencies.csv       — per-CWE feature presence rates")
print("   cwe_feature_discriminability.csv  — features ranked by mean |Δ| across all pairs")
print("   cwe_top_features_per_pair.csv     — top 20 features per confused pair")

✅ Exported:
   cwe_pattern_analysis_master.csv   — full (pair × feature) comparison table
   cwe_feature_frequencies.csv       — per-CWE feature presence rates
   cwe_feature_discriminability.csv  — features ranked by mean |Δ| across all pairs
   cwe_top_features_per_pair.csv     — top 20 features per confused pair
